<a href="https://colab.research.google.com/github/Harshit-Singhal-0100/datascience/blob/main/data_science_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ****TWEET SENTIMENTAL ANALYSIS****
data set from kaggle api and dataset url = "https://www.kaggle.com/datasets/kazanova/sentiment140/data"


In [ ]:
! pip install kaggle

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other


In [ ]:
from zipfile import ZipFile

dataset = '/content/sentiment140.zip'

with ZipFile(dataset, 'r') as zip_ref:
    zip_ref.extractall()

print("Extraction complete.")


Extraction complete.


**Requirements import file **

In [ ]:
# important import  file for the  tweet sentimental analysis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize
from textblob import TextBlob
from wordcloud import WordCloud
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import tweepy


In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

# **DATA PROCESSING **
1. remove stopwords

In [ ]:
twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding='ISO-8859-1')
# use encoding other than utf-8 to work properly
twitter_data.head()


,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [ ]:
twitter_data.shape

(1599999, 6)

challenge : data is in the row file

            
              1 target: the polarity of the tweet (0 = negative, 2 = neutral, 4 = positive
              2 ids: The id of the tweet ( 2087)
              3 date: the date of the tweet (Sat May 16
                23:58:44 UTC 2009)
              4 flag: The query (lyx). If there is no  
                 query, then this value is NO_QUERY.
              5 user: the user that tweeted (robotickilldozr)
              6 text: the text of the tweet (Lyx is cool)
            
solution:

    rename the row and column : for better visulization  

In [ ]:
col_name = ['target','ids','date','flag','user','text']
twitter_data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv',names = col_name, encoding='ISO-8859-1')
# use encoding other than utf-8 to work properly



twitter_data.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [ ]:
twitter_data.shape


(1600000, 6)

In [ ]:
# checking null values
twitter_data.isnull().sum()

,0
target,0
ids,0
date,0
flag,0
user,0
text,0


In [ ]:
twitter_data['target'].value_counts()

,count
target,
0,800000
4,800000


In [ ]:
# convert target 4->1
twitter_data.replace({'target':{4:1}},inplace=True)

In [ ]:
twitter_data['target'].value_counts()

,count
target,
0,800000
1,800000


now we have tyhe data in the target as 0 amnd 1 only
1-> positive comments
0-> negative


# ***Steaming***

---
process of changing the similar words to a single meaning <br>
Ex-> act is use for actor actress acting ...
      draw is use for drawing , designer ...
      conversion of acteress , actor and acting into act is steaming


In [ ]:
port_stem = PorterStemmer()

In [ ]:
def steaming(content):
    steamed_content = re.sub('[^a-zA-Z]', ' ', content)
    steamed_content = steamed_content.lower()
    words = word_tokenize(steamed_content)
    stemmed_words = [port_stem.stem(word) for word in words if word not in stopwords.words('english')]
    return ' '.join(stemmed_words)

In [48]:
twitter_data['steamed_content'] = twitter_data['text'].apply(steaming)


KeyboardInterrupt: 

In [49]:
twitter_data.head()

,target,ids,date,flag,user,text,steamed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see


In [50]:
twitter_data.to_csv('prcessed.csv', index=False)


In [ ]:
print(twitter_data['steamed_content'])

0          switchfoot http twitpic com zl awww bummer sho...
1          upset updat facebook text might cri result sch...
2          kenichan dive mani time ball manag save rest g...
3                            whole bodi feel itchi like fire
4                              nationwideclass behav mad see
                                 ...                        
1599995                           woke school best feel ever
1599996    thewdb com cool hear old walt interview http b...
1599997                         readi mojo makeov ask detail
1599998    happi th birthday boo alll time tupac amaru sh...
1599999    happi charitytuesday thenspcc sparkschar speak...
Name: steamed_content, Length: 1600000, dtype: object


**DATA SEPRATION:😊😊**

x target
y steamed data

In [51]:
x = twitter_data['steamed_content'].values
y = twitter_data['target'].values


In [ ]:
print(x)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


In [ ]:
print(y)

[0 0 0 ... 1 1 1]


**SPLITING DATA:😊😊**
TRANING and TESTING
x target
y steamed data

In [31]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify = y ,random_state=2)

In [32]:
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)


(1280000,) (1280000,) (320000,) (320000,)


In [34]:
print(x_train)

['watch saw iv drink lil wine' 'hatermagazin'
 'even though favourit drink think vodka coke wipe mind time think im gon na find new drink'
 ... 'eager monday afternoon'
 'hope everyon mother great day wait hear guy store tomorrow'
 'love wake folger bad voic deeper']


In [35]:
print(x_test)

['mmangen fine much time chat twitter hubbi back summer amp tend domin free time'
 'ah may show w ruth kim amp geoffrey sanhueza'
 'ishatara mayb bay area thang dammit' ...
 'destini nevertheless hooray member wonder safe trip' 'feel well'
 'supersandro thank']


In [36]:
# convert text into numeric form
vectorizer = TfidfVectorizer()
x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)

In [37]:
print(x_train)


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9489162 stored elements and shape (1280000, 461486)>
  Coords	Values
  (0, 436711)	0.27259876264838384
  (0, 354541)	0.3588091611460021
  (0, 185192)	0.5277679060576009
  (0, 109305)	0.3753708587402299
  (0, 235043)	0.41996827700291095
  (0, 443064)	0.4484755317023172
  (1, 160635)	1.0
  (2, 109305)	0.45280118482452514
  (2, 124483)	0.18661240251230476
  (2, 407299)	0.18451939024838399
  (2, 129410)	0.2867419530138451
  (2, 406397)	0.3166375845052528
  (2, 433558)	0.32512419345325
  (2, 77928)	0.30853680088497654
  (2, 443428)	0.33025302483687713
  (2, 266727)	0.23791347672998542
  (2, 409141)	0.14960586122844477
  (2, 178060)	0.1596736064437859
  (2, 150654)	0.18506405974027818
  (2, 282873)	0.165740495217716
  (2, 132310)	0.20010573505848944
  (2, 288468)	0.165559978152293
  (3, 406397)	0.29100963993007384
  (3, 158710)	0.44678357332595436
  (3, 151769)	0.2703326842167974
  :	:
  (1279996, 318301)	0.21254698865277744
  (12

In [38]:
print(x_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2298176 stored elements and shape (320000, 461486)>
  Coords	Values
  (0, 15110)	0.1719352837797837
  (0, 31168)	0.1624772418052177
  (0, 67827)	0.26800375270827315
  (0, 106068)	0.36555450010904555
  (0, 132363)	0.255254889555786
  (0, 138163)	0.23688292264071406
  (0, 171377)	0.2805816206356074
  (0, 271014)	0.45356623916588285
  (0, 279080)	0.17825180109103442
  (0, 388346)	0.2198507607206174
  (0, 398904)	0.34910438732642673
  (0, 409141)	0.3143047059807971
  (0, 420982)	0.17915624523539805
  (1, 6463)	0.30733520460524466
  (1, 15110)	0.211037449588008
  (1, 145392)	0.575262969264869
  (1, 217561)	0.40288153995289894
  (1, 256775)	0.28751585696559306
  (1, 348133)	0.4739279595416274
  (1, 366201)	0.24595562404108307
  (2, 22532)	0.3532582957477176
  (2, 34401)	0.37916255084357414
  (2, 89447)	0.36340369428387626
  (2, 183311)	0.5892069252021465
  (2, 256832)	0.2564939661498776
  :	:
  (319994, 443792)	0.2782185641032538


# **Training MACHINE LEARNING MODEL**

---
logistic regression



In [39]:
model = LogisticRegression(max_iter=1000)

In [40]:
model.fit(x_train,y_train)


LogisticRegression(max_iter=1000)

DATA IS TRAINED NOW CHECK THE ACCURACY score and test this out


In [53]:
# training data
x_train_prediction = model.predict(x_train)
training_data_accrocy = accuracy_score(y_train,x_train_prediction)

In [54]:
print('ACCURACY of training DATA: ' , training_data_accrocy*100)

ACCURACY of training DATA:  79.56289062500001


In [55]:
#  test data
x_test_prediction = model.predict(x_test)
testing_data_accrocy = accuracy_score(y_test,x_test_prediction)
print('ACCURACY of TEST DATA: ' , testing_data_accrocy*100)

ACCURACY of TEST DATA:  77.6465625


**ACCUACY: 77.6455625**

In [44]:
import pickle

In [47]:
filename = 'trained_model.pkl'
pickle.dump(twitter_data, open(filename, 'wb'))
filename = 'trained_model.sav'
pickle.dump(model, open(filename, 'wb'))


In [46]:
# future pridiction frfom the sam=ved and pre trained model
loaded_model = pickle.load(open('/content/trained_model.sav','rb'))
loaded_model1 = pickle.load(open('/content/trained_model.pkl','rb'))

FileNotFoundError: [Errno 2] No such file or directory: '/content/trained_model.sav'

In [ ]:
x_new = x_test[200]
print(y_test[200])

prediction = model.predict(x_new)
print(prediction)

if (prediction[0] == 0):
  print("negative tweet")

else:
  print('positive tweet')


In [56]:
x_new = "It seems that nothing ever goes as planned"
print(y_test[3])

x_new_vec = vectorizer.transform([x_new])

prediction = model.predict(x_new_vec)
print(prediction)

if prediction[0] == 0:
    print("negative tweet")
else:
    print("positive tweet")


0
[1]
positive tweet
